# Step 1: Independent Reference Construction

This notebook is paired with `step1_indep_reference_construction.R` and is intended to be opened from the `scripts/` folder. The full R construction code is copied below by dataset section. Outputs are written with explicit relative paths under `../Indep_scReference/<ref>/` and, for BLUE/scTAPE inputs, under `../scRNA_datasets/<ref>/`.

The older per-reference scripts under `../Indep_scReference/<ref>/` are kept for now and are not removed by this workflow.


In [ ]:
from pathlib import Path

scripts_dir = Path.cwd()
print('current folder:', scripts_dir)
print('expected helper:', Path('../DALE_Eval/modules/reference_prep_helpers.R').resolve())
print('output root:', Path('../Indep_scReference').resolve())
print('scRNA root:', Path('../scRNA_datasets').resolve())


## R setup shared by all sections


## BRCA_Wu2021

Output folder for this section:

```text
../Indep_scReference/BRCA_Wu2021/
```


### 1. Source loading and preprocessing


### 2. InstaPrism reference


### 3. cell type marker genes from pseudobulk-DE analysis


### 4. cbsx reference


### 5. bNIND referernce


### 6. ENIGMA reference


### 7. Output validation


## CRC_Lee2020

Output folder for this section:

```text
../Indep_scReference/CRC_Lee2020/
```


### 1. Source loading and preprocessing


### 2. InstaPrism reference


### 3. cell type marker genes from pseudobulk-DE analysis


### 4. cbsx reference


### 5. bNIND referernce


### 6. ENIGMA reference


### 7. Output validation


## LUAD_Laughney2020

Output folder for this section:

```text
../Indep_scReference/LUAD_Laughney2020/
```


### 1. Source loading and preprocessing


### 2. InstaPrism reference


### 3. bMIND reference


### 4. ENIGMA reference


### 5. cell type marker genes from pseudobulk-DE analysis


### 6. cbsx signatures


### 7. Output validation


## PBMC_AIDA2024 / PBMC_refined_AIDA2024

Output folder for this section:

```text
../Indep_scReference/PBMC_AIDA2024/
../Indep_scReference/PBMC_refined_AIDA2024/
../scRNA_datasets/PBMC_AIDA2024/PBMC_AIDA2024_processed.h5ad
```


### 1. Source loading and preprocessing


### 2. BLUE/scTAPE processed h5ad export, Python only


In [ ]:
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp

source_h5ad = Path("../../../InstaPrismExtension/scRNA_resource/Cellxgene/PBMC_AIDA/85ca63ad-10c2-4c9e-9f76-a52568b294f1.h5ad")
out_dir = Path("../scRNA_datasets/PBMC_AIDA2024")
out_h5ad = out_dir / "PBMC_AIDA2024_processed.h5ad"
coarse_yaml = Path("../Indep_scReference/PBMC_AIDA2024/celltype_mapping.yaml")
refined_yaml = Path("../Indep_scReference/PBMC_refined_AIDA2024/celltype_mapping.yaml")

ct_map_coarse = {
    "T_cell": [
        "CD4+_T", "CD4+_T_cm", "CD4+_T_cyt", "CD4+_T_em", "CD4+_T_naive",
        "CD8+_T", "CD8+_T_GZMB+", "CD8+_T_GZMK+", "CD8+_T_naive", "MAIT",
        "dnT", "gdT", "T", "Treg",
    ],
    "B_cell": ["atypical_B", "B", "IGHMhi_memory_B", "IGHMlo_memory_B", "naive_B", "Plasma_B"],
    "myeloid": ["CD14+_Monocyte", "CD16+_Monocyte", "Monocyte", "cDC", "cDC1", "cDC2", "DC"],
    "NK": ["CD16+_NK", "CD56+_NK", "NK"],
}

ct_map_refined = {
    "CD4_T": ["CD4+_T", "CD4+_T_cm", "CD4+_T_cyt", "CD4+_T_em", "CD4+_T_naive", "Treg"],
    "CD8_T": ["CD8+_T", "CD8+_T_GZMB+", "CD8+_T_GZMK+", "CD8+_T_naive", "MAIT"],
    "gd_T": ["gdT"],
    "B_cell": ["atypical_B", "B", "IGHMhi_memory_B", "IGHMlo_memory_B", "naive_B"],
    "Plasma_B": ["Plasma_B"],
    "CD14_Mono": ["CD14+_Monocyte"],
    "CD16_Mono": ["CD16+_Monocyte"],
    "DC": ["cDC", "cDC1", "cDC2", "DC"],
    "NK": ["CD16+_NK", "CD56+_NK", "NK"],
}

def recode(values, mapping):
    lookup = {fine: coarse for coarse, fines in mapping.items() for fine in fines}
    return pd.Series(values, index=values.index, dtype="object").map(lookup)

def write_self_mapping(path, labels):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w") as handle:
        handle.write("# BLUE/scTAPE mapping generated from step1_indep_reference_construction.ipynb\n")
        handle.write("# Keys are method output classes; values are labels in the selected h5ad obs column.\n")
        for label in labels:
            handle.write(f"{label}:\n")
            handle.write(f"  - '{label}'\n")

adata = ad.read_h5ad(source_h5ad)
print("source:", adata.shape)

adata = adata[adata.obs["Country"].astype(str) == "South_Korea", :].copy()
print("South_Korea:", adata.shape)

feature_name = adata.var["feature_name"].astype(str)
gene_keep = np.asarray((adata.X > 0).sum(axis=0)).ravel() >= 100
gene_keep &= ~feature_name.duplicated().to_numpy()

cell_keep = (
    (adata.obs["nCount_RNA"] >= 1000)
    & (adata.obs["nFeature_RNA"] >= 500)
    & (~adata.obs["author_cell_type"].astype(str).isin(["ILC", "Platelet", "RBC", "pDC"]))
)

adata = adata[cell_keep.to_numpy(), gene_keep].copy()
adata.var_names = adata.var["feature_name"].astype(str)
print("after QC:", adata.shape)

obs_source = adata.obs.copy()
coarse = recode(obs_source["author_cell_type"].astype(str), ct_map_coarse)
refined = recode(obs_source["author_cell_type"].astype(str), ct_map_refined)
keep_mapped = coarse.notna() & refined.notna() & obs_source["donor_id"].notna()

adata = adata[keep_mapped.to_numpy(), :].copy()
obs_source = obs_source.loc[adata.obs_names]
coarse = coarse.loc[adata.obs_names]
refined = refined.loc[adata.obs_names]
print("after BLUE/scTAPE mapping filter:", adata.shape)
print("unique sample ids:", obs_source["donor_id"].astype(str).nunique())

X = adata.X.copy()
if sp.issparse(X):
    X = X.tocsr()
    X.data = np.expm1(X.data)
else:
    X = np.expm1(np.asarray(X))
    X = sp.csr_matrix(X)

obs = pd.DataFrame(
    {
        "sample": obs_source["donor_id"].astype(str).to_numpy(),
        "cell_type": coarse.astype(str).to_numpy(),
        "cell_type_refined": refined.astype(str).to_numpy(),
        "author_cell_type_original": obs_source["author_cell_type"].astype(str).to_numpy(),
        "donor_id_original": obs_source["donor_id"].astype(str).to_numpy(),
    },
    index=adata.obs_names.astype(str),
)
var = pd.DataFrame(
    {"gene_symbol": adata.var_names.astype(str)},
    index=adata.var_names.astype(str),
)

ad_blue = ad.AnnData(X=X, obs=obs, var=var)
out_dir.mkdir(parents=True, exist_ok=True)
ad_blue.write_h5ad(out_h5ad)

write_self_mapping(coarse_yaml, ["T_cell", "B_cell", "myeloid", "NK"])
write_self_mapping(refined_yaml, ["CD4_T", "CD8_T", "gd_T", "B_cell", "Plasma_B", "CD14_Mono", "CD16_Mono", "DC", "NK"])

print("wrote:", out_h5ad)
print("wrote:", coarse_yaml)
print("wrote:", refined_yaml)
print(ad_blue.obs[["sample", "cell_type", "cell_type_refined", "author_cell_type_original"]].head())


### 3. PBMC_AIDA2024


### 4. 01. bMIND reference


### 5. 02. ENIGMA reference


### 6. 03. cell type marker genes from pseudobulk-DE analysis


### 7. 04. InstaPrism reference


### 8. 05. cbsx reference


### 9. PBMC_refined_AIDA2024


### 10. 01. bMIND reference


### 11. 02. ENIGMA reference


### 12. 03. cell type marker genes from pseudobulk-DE analysis


### 13. 04. InstaPrism reference


### 14. 05. cbsx reference


### 15. Output validation


## ROSMAP_MultiregionAD_Mathys2024

Output folder for this section:

```text
../Indep_scReference/ROSMAP_MultiregionAD_Mathys2024/
../scRNA_datasets/ROSMAP_MultiregionAD_Mathys2024/ROSMAP_MultiregionAD_Mathys2024_processed.h5ad
```


### 1. Source loading and preprocessing


### 2. BLUE/scTAPE processed h5ad export


### 3. BLUE/scTAPE celltype mapping


### 4. bMIND reference


### 5. ENIGMA reference


### 6. cell type marker genes from pseudobulk-DE analysis


### 7. cbsx signatures


### 8. InstaPrism reference


### 9. Output validation


## Final Readiness Summary


In [ ]:
import pandas as pd
from pathlib import Path

required_ref_files = [
    'refPhi.RDS',
    'cbsx_sig.txt',
    'bMIND_profile.csv',
    'rowMeans_sig.csv',
    'limma_top_genes.csv',
]

def reference_status(ref_name):
    ref_dir = Path('../Indep_scReference') / ref_name
    return pd.DataFrame([
        {'indep_ref': ref_name, 'file': name, 'exists': (ref_dir / name).exists()}
        for name in required_ref_files
    ])

refs = [
    'BRCA_Wu2021',
    'CRC_Lee2020',
    'LUAD_Laughney2020',
    'PBMC_AIDA2024',
    'PBMC_refined_AIDA2024',
    'ROSMAP_MultiregionAD_Mathys2024',
]
ref_summary = pd.concat([reference_status(ref) for ref in refs], ignore_index=True)
ref_summary.pivot(index='indep_ref', columns='file', values='exists')
